In [1]:
import numpy as np
import pinocchio as pin
import eigenpy
from scipy.spatial.transform import Rotation
import mujoco

In [ ]:
target_quat = np.array([0., 1., 0., 0.])
euler = np.array([np.deg2rad(-10), 0, 0])

rot_slope = Rotation.from_euler('xyz', euler)
rot_target = Rotation.from_quat(target_quat, scalar_first=True)
result_rot = rot_slope * rot_target
result_quat = result_rot.as_quat(scalar_first=False)

In [4]:
rot_slope = Rotation.from_euler('xyz', euler)

# target_quat is (w, x, y, z) → convert to SciPy order (x, y, z, w)
rot_target = Rotation.from_quat(np.roll(target_quat, -1))

result_rot = rot_slope * rot_target

# result back to (w, x, y, z)
result_quat = np.roll(result_rot.as_quat(), 1)

In [ ]:
target_quat = np.array([target_quat])

In [17]:
(rot_slope * rot_target).as_quat(scalar_first=True)

array([0.08715574, 0.9961947 , 0.        , 0.        ])

In [5]:
result_quat

array([0.08715574, 0.9961947 , 0.        , 0.        ])

In [ ]:
target_quat = np.array([0., 1., 0., 0.])
quat_slope = np.zeros(4)
mujoco.mju_euler2Quat(quat_slope, euler, 'XYZ')
mujoco.mju_mulQuat(target_quat, quat_slope, target_quat)

In [21]:
target_quat # q = (w, x, y, z)

array([0.08715574, 0.9961947 , 0.        , 0.        ])

Results are same

angle between 2 quaternions

In [9]:
import numpy as np
import pinocchio as pin
import eigenpy
from scipy.spatial.transform import Rotation
import mujoco

In [13]:
target_quat = np.array([0., 1., 0., 0.])
euler = np.array([np.deg2rad(-10), 0, 0])
current_r = Rotation.random()
current_mat = current_r.as_matrix()
rot_target = Rotation.from_quat(target_quat, scalar_first=True)

In [14]:
def compute_ee_pose_error(target_pos, current_pos, target_quat, current_mat, Kpos=0.95):
    twist = np.zeros(6)
    site_quat = np.zeros(4)
    site_quat_conj = np.zeros(4)
    error_quat = np.zeros(4)
    # Kpos Gains for the twist computation. These should be between 0 and 1. 0 means no
    # movement, 1 means move the end-effector to the target in one integration step.
    # Gain for the orientation component of the twist computation. This should be
    # between 0 and 1. 0 means no movement, 1 means move the end-effector to the target
    # orientation in one integrati on step.
    Kori: float = 0.95

    dx = target_pos - current_pos
    twist[:3] = Kpos * dx
    mujoco.mju_mat2Quat(site_quat, current_mat)
    mujoco.mju_negQuat(site_quat_conj, site_quat)
    mujoco.mju_mulQuat(error_quat, target_quat, site_quat_conj)
    mujoco.mju_quat2Vel(twist[3:], error_quat, 1.0)
    # twist[3:] *= Kori
    return twist

compute_ee_pose_error(
    target_pos=np.array([0,0,0]),
    current_pos=np.array([0,0,0]),
    target_quat=target_quat,
    current_mat=current_mat.flatten()
)

array([ 0.        ,  0.        ,  0.        , -0.5668289 ,  0.55409753,
        1.13588067])

In [15]:
R_error = rot_target * current_r.inv()
R_error.as_rotvec()

array([-0.5668289 ,  0.55409753,  1.13588067])

array([ 0.        ,  0.        ,  0.        , -0.09745943, -1.21782453,
        1.66682929])

euler to quat

In [2]:
euler = np.array([np.deg2rad(-10), 0, 0])

In [7]:
quat_slope = np.zeros(4)
mujoco.mju_euler2Quat(quat_slope, euler, 'XYZ')
quat_slope

array([ 0.9961947 , -0.08715574,  0.        ,  0.        ])

In [8]:
rot_slope = Rotation.from_euler('xyz', euler)
rot_slope.as_quat(scalar_first=True)

array([ 0.9961947 , -0.08715574,  0.        ,  0.        ])

In [6]:
target_pos = np.array([0.6, 0.00173648, 0.45984808])
current_pos = np.array([0.60017669, 0.002716, 0.48241737])
np.linalg.norm(current_pos - target_pos)

np.float64(0.02259122683456126)